# wk4 - More Apache Spark - Part 1

In [ ]:
# Week 7
# Demo code - Part 1 - PySpark

In [ ]:
# 1 Create a connection
# get the details of the local Spark environment
#import findspark
#findspark.init()
from pyspark.sql import SparkSession

In [ ]:
#Create a Spark Session
spark = SparkSession.builder \
        .appName('ProgBigData-Spark - wk4 Demo 1') \
        .master('local[*]') \
        .getOrCreate()

# spark.stop()

# Load data from Docker

In [ ]:
!ls

In [ ]:
dataPath = "datasets/diamonds.csv"
#default is Parquet
diamonds1 = spark.read.format("csv") \
  .load(dataPath)

display(diamonds1)
#look at the column names!

In [ ]:
diamonds1.show()
#look at the column names!

In [ ]:
#Check the number of partitions
print(diamonds1.rdd.getNumPartitions())

#We should get 1 (one) partition - this is default for reading a file

In [ ]:
from pyspark.sql import Row

sdf2 = spark.createDataFrame([
    Row(a=1, b=2., c='John'),
    Row(a=2, b=3., c='Patrick'),
    Row(a=4, b=5., c='Seamus'),
    Row(a=4, b=5., c='George'),
    Row(a=4, b=5., c='Sean'),
    Row(a=4, b=5., c='Mary'),
    Row(a=4, b=5., c='Ann'),
    Row(a=4, b=5., c='Jane'),
    Row(a=4, b=5., c='Deirdre'),
    Row(a=4, b=5., c='Amanda')
], schema='a long, b double, c string')
sdf2  #spark DF

In [ ]:
#Get the number of partitions
print(sdf2.rdd.getNumPartitions())

#We should see 8 - this is default within Spark  - In Colab this is 2
#This data is distributed over 8 partitions in-memory - All work will be distributed over these 8 partitions

# But how/why difference with reading a file?  How can we change that?

In [ ]:
# Read the file again and repartition it

dataPath = "datasets/diamonds.csv"
#default is Parquet
diamonds2 = spark.read.format("csv") \
  .load(dataPath) \
  .repartition(8)

#The last part of the Read command divides the data into 8 partitions

In [ ]:
#Print number of partitions
print(diamonds2.rdd.getNumPartitions())

#we should now have 8 partitions

In [ ]:
#Get storage level information
diamonds2.storageLevel

In [ ]:
#change the storage level to in-memory
import pyspark

diamonds2_persist = diamonds2.persist(pyspark.StorageLevel.MEMORY_ONLY)
diamonds2.storageLevel

In [ ]:
#I could have just Cashed the results
diamonds2.cache()

#with lazy evaluation the DF will not be cached untile we perform an Action on it.
# e.g.
diamonds2.count()

In [ ]:
diamonds2.show()

# 3 Transformations and Actions

In [ ]:
#Transformations & Actions
#  Just a Simple example(s) to get you started    -- More on these Next Week
#
# Transformations (lazy evaluation) include:
#    - select
#    - distinct
#    - groupBy
#    - sum
#    - orderBy
#    - filter
#    - limit
#    - orderBy
#    - sort
#    - ...

# Actions - Force the Transformations to be executed and data created -- More on these next week
#    - show
#    - count
#    - collect
#    - save
#
#Check out the Documentation for full list - https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.sql.DataFrame.html
#
# Let's work through a more detailed example

# Data sets from the book (Learning Spark) can be found at
# https://github.com/databricks/LearningSparkV2

In [ ]:
#Define the record structure
from pyspark.sql.types import *

fire_schema = StructType([StructField('CallNumber', IntegerType(), True),
   StructField('UnitID', StringType(), True),
   StructField('IncidentNumber', IntegerType(), True),
   StructField('CallType', StringType(), True),
   StructField('CallDate', StringType(), True),
   StructField('WatchDate', StringType(), True),
   StructField('CallFinalDisposition', StringType(), True),
   StructField('AvailableDtTm', StringType(), True),
   StructField('Address', StringType(), True),
   StructField('City', StringType(), True),
   StructField('Zipcode', IntegerType(), True),
   StructField('Battalion', StringType(), True),
   StructField('StationArea', StringType(), True),
   StructField('Box', StringType(), True),
   StructField('OriginalPriority', StringType(), True),
   StructField('Priority', StringType(), True),
   StructField('FinalPriority', IntegerType(), True),
   StructField('ALSUnit', BooleanType(), True),
   StructField('CallTypeGroup', StringType(), True),
   StructField('NumAlarms', IntegerType(), True),
   StructField('UnitType', StringType(), True),
   StructField('UnitSequenceInCallDispatch', IntegerType(), True),
   StructField('FirePreventionDistrict', StringType(), True),
   StructField('SupervisorDistrict', StringType(), True),
   StructField('Neighborhood', StringType(), True),
   StructField('Location', StringType(), True),
   StructField('RowID', StringType(), True),
   StructField('Delay', FloatType(), True)])

# Use the DataFrameReader interface to read a CSV file

sf_fire_file = "datasets/sf-fire-calls.csv"

fire_df = spark.read.csv(sf_fire_file, header=True, schema=fire_schema)

In [ ]:
#Let's cache the data. This will make the subsequent commands run quicker
fire_df.cache()
fire_df.show(10)

In [ ]:
#Did you notice in the above example, some columns do not display all the values
# They display a subset and then print ...
#
#To display the full values, use False,   or truncate=False
fire_df.show(10, False)

In [ ]:
#Projections and Filters
# Projection = specify that parts to return - use  select()
# Filter = an expression used to return a subset of the records - use  filter()  or  where()

# Example - create a new dataframe to contain IncidentNumber, Available DtTm, CallType,
#              where CallType != Medical Incident

from pyspark.sql.functions import *

few_fire_df = (fire_df.select("IncidentNumber", "AvailableDtTm", "CallType")
              .where(col("CallType") != "Medical Incident"))

few_fire_df.show(10, False)
# Change True to False, see what happens.  Above is same as having    truncate=False

In [ ]:
#What if we want to know how many distinct CallTypes values
fire_df.select("CallType").where(col("CallType").isNotNull()).distinct().count()

#Q: Do we need to use  .isNotNull() in the above command?  Explore this.

In [ ]:
#An Alternative - Using Aggregrate functions
# Allows for some formatting of output/result set
fire_df.select("CallType") \
       .where(col("CallType").isNotNull()) \
       .agg(count_distinct("CallType").alias("DistinctCallTypes")) \
       .show()

#We also renamed a column uing  alias.  An alternative to this will be shown later

In [ ]:
#Exploring the data - What are the distinct CallTypes
fire_df.select("CallType") \
       .distinct() \
       .show(10, False)

#       .orderBy("CallType", ascending=True) \

#change  ascending  to False, see what happens

In [ ]:
#Exploring the data - What are the distinct CallTypes
fire_df.select("CallType") \
       .distinct() \
       .orderBy("CallType", ascending=True) \
       .show(10, False)



In [ ]:
#Remember the question I asked earlier, if  isNotNull() needed?
#We can find out using this
fire_df.select("CallType").where(col("CallType").isNull()) \
.distinct() \
.show(10, False)

In [ ]:
#Aggregations - grounBy, Count, orderBy
fire_df.select("CallType").where(col("CallType").isNotNull()) \
       .groupBy("CallType") \
       .count() \
       .orderBy("count", ascending=False) \
       .show(n=10, truncate=False)

In [ ]:
# Get summary statistics (Count, Mean, Stddev, Min, Max). -NB: This takes several minutes to run
fire_df.describe().show()

In [ ]:
# Q: Can you find a better way to display the data above?
fire_df.describe().show(3,vertical=True)


In [ ]:
fire_df.summary().show()

In [ ]:
# Q: Can you find a better way to display the data above?
fire_df.summary().show(3, vertical=True)


In [ ]:
#Renaming and modifying the contents are common steps
#Renaming columns
new_fire_df = fire_df.withColumnRenamed("Delay", "ResponseDelayedinMins")

new_fire_df.select("ResponseDelayedinMins") \
           .where(col("ResponseDelayedinMins") > 5) \
           .show(5, False)

In [ ]:
#Modifying data and Droping Columns
fire_ts_df = new_fire_df \
             .withColumn("IncidentDate", to_timestamp(col("CallDate"), "MM/dd/yyyy")) \
             .drop("CallDate") \
             .withColumn("OnWatchDate", to_timestamp(col("WatchDate"), "MM/dd/yyyy")) \
             .drop("WatchDate") \
             .withColumn("AvailableDtTS", to_timestamp(col("AvailableDtTm"),"MM/dd/yyyy hh:mm:ss a")) \
             .drop("AvailableDtTm")

fire_ts_df.select("IncidentDate", "OnWatchDate", "AvailableDtTS") \
          .show(5, False)

#Q: Where the columns dropped? How can you check and verify this?
# fire_ts_df.select("CallDate", "WatchDate", "AvailableDtTm")

In [ ]:
#Now the data has been transformed (string -> timestamp), it can be queried using function
(fire_ts_df
.select(year('IncidentDate'))
.distinct()
.orderBy(year('IncidentDate'))
.show())

In [ ]:
#Q: modified this to include number of incidents
(fire_ts_df
 .select(year("IncidentDate").alias("Year"))
 .groupBy("Year")
 .agg(count("Year").alias("Number_of_incidents")) 
 .orderBy("Number_of_incidents", ascending=False)
 .show())

In [ ]:
#More Aggregations
#Calculate some Statistics
import pyspark.sql.functions as F

new_fire_df.select(F.sum("NumAlarms"), F.avg("ResponseDelayedinMins"), \
           F.min("ResponseDelayedinMins"), F.max("ResponseDelayedinMins")) \
.show()

# 4 Joining Spark Data Frames and Processing Data

In [ ]:
#Joining dataframes/data sets

#setup 2 dataframes
#1st a Customer df
cust_df = spark.createDataFrame([
    Row(A=1, County='D', CName='John'),
    Row(A=2, County='C', CName='Patrick'),
    Row(A=3, County='D', CName='Seamus'),
    Row(A=4, County='CN', CName='George'),
    Row(A=5, County='MH', CName='Sean'),
    Row(A=6, County='C', CName='Mary'),
    Row(A=7, County='MN', CName='Ann'),
    Row(A=8, County='MN', CName='Jane'),
    Row(A=9, County='TN', CName='Deirdre'),
    Row(A=10, County='D', CName='Amanda'),
    Row(A=11, County='KY', CName='Richard')
])
cust_df  #spark DF

In [ ]:
#setup a County df
county_df = spark.createDataFrame([
    Row(County='D', CountyName='Dublin'),
    Row(County='C', CountyName='Cork'),
    Row(County='CN', CountyName='Cavan'),
    Row(County='MH', CountyName='Meath'),
    Row(County='MN', CountyName='Dublin'),
    Row(County='WW', CountyName='Wicklow'),
    Row(County='G', CountyName='Galway')
])
county_df  #spark DF

#Now let's try different ways of joining the data

In [ ]:
#Left Join
left_join = cust_df.join(county_df, cust_df.County == county_df.County, how='left').sort(cust_df.A)
left_join.show()

#Notice the Nulls - Nulls when no corresponding data in County df
#This is a Major Data Quality issue

In [ ]:
#Inner Join
inner_join = cust_df.join(county_df, cust_df.County == county_df.County, how='inner').sort(cust_df.A)
inner_join.show()

#The nulls have disappeared
#But some data is missing. Why?

In [ ]:
#Right join
right_join = cust_df.join(county_df, cust_df.County == county_df.County, how='right').sort(cust_df.A)
right_join.show()

#We have Nulls again - This time in a different location
#Again a Data Quality issue

In [ ]:
#Outer join
outer_join = cust_df.join(county_df, cust_df.County == county_df.County, how='outer').sort(cust_df.A)
outer_join.show()

#We get nulls on both sides
#Data Quality issue

In [ ]:
#Left join with Filter + Drop column
# where cust_df.County = D
# drop column cust_df.County  - to remove duplicate columns being displayed
join = cust_df.join(county_df, cust_df.County == county_df.County, how='left').drop(cust_df.County).where(cust_df.County == 'D').show()


In [ ]:
#Left join with filter + drop  +  column rename
join = cust_df.withColumnRenamed('CName', 'Customer_Name').join(county_df, cust_df.County == county_df.County, how='left').drop(cust_df.County).where(cust_df.County == 'D').show()


In [ ]:
#Left join with filter + drop  +  column rename + Projection
# Projection = subset of columns
join = cust_df.withColumnRenamed('CName', 'Customer_Name').join(county_df, cust_df.County == county_df.County, how='left').drop(cust_df.County).where(cust_df.County == 'D').select(['Customer_Name', 'CountyName']).show()


# 5 User Defined Functions

In [ ]:
#UDF - User Defined Functions

#display data set
left_join.show()

In [ ]:
#Create UDF to change value in one column

from pyspark.sql.functions import UserDefinedFunction
from pyspark.sql.types import StringType

#define UDF  :  multiply column value by 10
udf = UserDefinedFunction(lambda x: x*10, StringType())
#Apply UDF
left_udf = left_join.withColumn('A', udf("A")).show()

#Add new column using UDF - could have renamed column
left_udf2 = left_join.withColumn('ID', udf("A")).show()

In [ ]:
#Alternative UDF
import pandas as pd
from pyspark.sql.functions import pandas_udf

#@pandas_udf('long')
def pd_udf_by_20(series: pd.Series) -> pd.Series:
    return series * 20

left_udf3 = left_join.select(pd_udf_by_20(left_join.A)).show()

left_udf4 = left_join.withColumn('A', pd_udf_by_20(left_join.A)).show()
